# 02_MODELISATION_BASELINES_MLFLOW

## Objectif

Ce notebook reprend une logique simple :

1. charger le dataset préparé ;
2. faire un split train / holdout ;
3. tester quelques modèles baseline faciles à lire ;
4. suivre proprement les runs dans MLflow ;
5. retenir un premier candidat pour l'étape d'optimisation.

L'idée est d'avoir un notebook lisible et pédagogique, sans déplacer
la complexité dans les cellules.


In [14]:
from dataclasses import dataclass
from pathlib import Path
from tempfile import TemporaryDirectory
import importlib
import json
import sys

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from IPython.display import display
from mlflow import MlflowClient
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import lightgbm as lgb
except ImportError:
    lgb = None

def resolve_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "Impossible de localiser la racine du projet contenant 'src/' et 'data/'."
    )

project_root = resolve_project_root(Path.cwd())

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.Fonctions_MODEL as fonctions_model
fonctions_model = importlib.reload(fonctions_model)

classification_metrics_at_threshold = fonctions_model.classification_metrics_at_threshold
evaluate_classifier_cv = fonctions_model.evaluate_classifier_cv
evaluate_holdout = fonctions_model.evaluate_holdout
flatten_cv_summary = fonctions_model.flatten_cv_summary
prepare_modeling_tables = fonctions_model.prepare_modeling_tables
sanitize_model_params = fonctions_model.sanitize_model_params
score_to_probability = fonctions_model.score_to_probability

processed_dir = project_root / "data" / "processed"
report_dir = processed_dir / "reports"
benchmark_report_dir = report_dir / "baseline_benchmark"
benchmark_report_dir.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 200)

random_state = 42
cv_folds = 3
business_fn_cost = 10.0
business_fp_cost = 1.0
train_metric_sample_size = 20_000
aligned_metric_keys = [
    "accuracy",
    "f1",
    "business_fbeta",
    "roc_auc",
    "pr_auc",
]

@dataclass(frozen=True)
class MlflowTrackingContext:
    client: MlflowClient
    tracking_uri: str
    artifact_location: str
    experiment_name: str

def configure_mlflow_tracking(
    project_root: Path,
    experiment_name: str = "P6_HOME_CREDIT_DEFAULT_RISK",
) -> MlflowTrackingContext:
    tracking_db_path = project_root / "mlflow.db"
    artifact_root = project_root / "mlartifacts"
    artifact_root.mkdir(parents=True, exist_ok=True)

    tracking_uri = f"sqlite:///{tracking_db_path.as_posix()}"
    artifact_location = artifact_root.resolve().as_uri()

    mlflow.set_tracking_uri(tracking_uri)
    client = MlflowClient()

    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        experiment_id = client.create_experiment(
            name=experiment_name,
            artifact_location=artifact_location,
        )
        experiment = client.get_experiment(experiment_id)

    mlflow.set_experiment(experiment_name)
    return MlflowTrackingContext(
        client=client,
        tracking_uri=tracking_uri,
        artifact_location=experiment.artifact_location,
        experiment_name=experiment_name,
    )

def resolve_lightgbm_device_type(random_state: int = 42) -> str:
    if lgb is None:
        return "cpu"

    probe_X = pd.DataFrame(
        np.random.default_rng(seed=random_state).random((512, 8), dtype=np.float32)
    )
    probe_y = pd.Series(
        np.random.default_rng(seed=random_state + 1).integers(0, 2, size=512)
    )

    try:
        probe_model = lgb.LGBMClassifier(
            objective="binary",
            n_estimators=5,
            random_state=random_state,
            device_type="gpu",
            verbosity=-1,
        )
        probe_model.fit(probe_X, probe_y)
        return "gpu"
    except Exception:
        return "cpu"

def build_simple_baseline_catalog() -> pd.DataFrame:
    rows = [
        {
            "model_name": "dummy_classifier",
            "family": "Baseline naïve",
            "benchmark_scope": "full_dataset",
            "why_included": "référence minimale pour montrer qu'une forte accuracy ne suffit pas",
        },
        {
            "model_name": "sgd_log_loss",
            "family": "Baseline linéaire",
            "benchmark_scope": "full_dataset",
            "why_included": "baseline linéaire scalable, mieux adaptée que LogisticRegression au volume du projet",
        },
        {
            "model_name": "hist_gradient_boosting",
            "family": "Booster scikit-learn",
            "benchmark_scope": "full_dataset",
            "why_included": "baseline tabulaire robuste, simple à comparer à LightGBM",
        },
    ]

    if lgb is not None:
        rows.append(
            {
                "model_name": "lightgbm_bonus",
                "family": "Booster bonus",
                "benchmark_scope": "full_dataset",
                "why_included": "benchmark bonus plus puissant et bon candidat pour l'optimisation",
            }
        )

    return pd.DataFrame(rows)

def build_simple_baseline_models(
    random_state: int = 42,
    lightgbm_device_type: str = "cpu",
) -> dict[str, object]:
    models: dict[str, object] = {
        "dummy_classifier": DummyClassifier(strategy="most_frequent"),
        "sgd_log_loss": Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "model",
                    SGDClassifier(
                        loss="log_loss",
                        alpha=1e-4,
                        max_iter=2_000,
                        tol=1e-3,
                        class_weight="balanced",
                        random_state=random_state,
                    ),
                ),
            ]
        ),
        "hist_gradient_boosting": HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            min_samples_leaf=100,
            random_state=random_state,
        ),
    }

    if lgb is not None:
        lightgbm_params = {
            "objective": "binary",
            "learning_rate": 0.05,
            "n_estimators": 300,
            "num_leaves": 31,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "class_weight": "balanced",
            "random_state": random_state,
            "verbosity": -1,
        }
        if lightgbm_device_type == "gpu":
            lightgbm_params["device_type"] = "gpu"

        models["lightgbm_bonus"] = lgb.LGBMClassifier(**lightgbm_params)

    return models

def log_dataframe_artifact(
    df: pd.DataFrame,
    filename: str,
    artifact_path: str = "tables",
) -> None:
    with TemporaryDirectory() as tmp_dir:
        csv_path = Path(tmp_dir) / filename
        df.to_csv(csv_path, index=False)
        mlflow.log_artifact(str(csv_path), artifact_path=artifact_path)

def build_train_test_metric_row(
    model_name: str,
    family: str,
    metadata: dict,
    threshold: float,
    train_metrics: dict,
    test_metrics: dict,
    summary_flat: dict,
) -> dict:
    row = {
        "model_name": model_name,
        "family": family,
        "benchmark_scope": metadata.get("benchmark_scope", "full_dataset"),
        "threshold": float(threshold),
    }

    for metric in aligned_metric_keys:
        row[f"train_{metric}"] = float(train_metrics.get(metric, np.nan))
        row[f"test_{metric}"] = float(test_metrics.get(metric, np.nan))

    row["train_business_cost_per_obs"] = float(
        train_metrics.get("business_cost_per_obs", np.nan)
    )
    row["test_business_cost_per_obs"] = float(
        test_metrics.get("business_cost_per_obs", np.nan)
    )

    # Colonnes CV conservées pour le pilotage et la compatibilité avec la suite.
    row["valid_business_cost_per_obs_mean"] = float(
        summary_flat.get("valid_business_cost_per_obs_mean", np.nan)
    )
    row["valid_fit_time_seconds_mean"] = float(
        summary_flat.get("valid_fit_time_seconds_mean", np.nan)
    )
    row["valid_business_fbeta_mean"] = float(
        summary_flat.get("valid_business_fbeta_mean", np.nan)
    )
    row["valid_pr_auc_mean"] = float(summary_flat.get("valid_pr_auc_mean", np.nan))
    row["valid_roc_auc_mean"] = float(summary_flat.get("valid_roc_auc_mean", np.nan))

    # Alias holdout conservés pour ne pas casser les artefacts déjà attendus ailleurs.
    row["holdout_accuracy"] = row["test_accuracy"]
    row["holdout_precision"] = float(test_metrics.get("precision", np.nan))
    row["holdout_recall"] = float(test_metrics.get("recall", np.nan))
    row["holdout_f1"] = row["test_f1"]
    row["holdout_business_fbeta"] = row["test_business_fbeta"]
    row["holdout_roc_auc"] = row["test_roc_auc"]
    row["holdout_pr_auc"] = row["test_pr_auc"]
    row["holdout_threshold"] = float(threshold)
    row["holdout_false_negatives"] = float(
        test_metrics.get("false_negatives", np.nan)
    )
    row["holdout_false_positives"] = float(
        test_metrics.get("false_positives", np.nan)
    )
    row["holdout_business_cost"] = float(test_metrics.get("business_cost", np.nan))
    row["holdout_business_cost_per_obs"] = row["test_business_cost_per_obs"]
    row["holdout_fit_time_seconds"] = float(
        test_metrics.get("fit_time_seconds", np.nan)
    )
    row["overfit_gap_pr_auc"] = float(
        summary_flat.get("train_pr_auc_mean", np.nan)
        - summary_flat.get("valid_pr_auc_mean", np.nan)
    )
    row["overfit_gap_roc_auc"] = float(
        summary_flat.get("train_roc_auc_mean", np.nan)
        - summary_flat.get("valid_roc_auc_mean", np.nan)
    )
    return row

def log_train_test_metrics(train_metrics: dict, test_metrics: dict) -> None:
    payload = {}
    for metric in aligned_metric_keys:
        if metric in train_metrics:
            payload[f"train_{metric}"] = float(train_metrics[metric])
        if metric in test_metrics:
            payload[f"test_{metric}"] = float(test_metrics[metric])

    if payload:
        mlflow.log_metrics(payload)

def run_simple_baseline_benchmark_with_mlflow(
    models: dict[str, object],
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_holdout: pd.DataFrame,
    y_holdout: pd.Series,
    processed_dir: Path,
    run_name: str,
    cv_folds: int,
    threshold: float,
    fn_cost: float,
    fp_cost: float,
    random_state: int,
    train_metric_sample_size: int | None,
    model_catalog: pd.DataFrame | None = None,
    tracking_tags: dict | None = None,
    extra_artifacts: list[Path] | None = None,
    log_candidate_model_name: str | None = None,
) -> tuple[pd.DataFrame, dict, dict]:
    leaderboard_rows = []
    fitted_models: dict[str, object] = {}
    tracking_tags = tracking_tags or {}
    extra_artifacts = extra_artifacts or []
    model_catalog = model_catalog.copy() if model_catalog is not None else pd.DataFrame()
    catalog_lookup = (
        model_catalog.set_index("model_name").to_dict(orient="index")
        if not model_catalog.empty
        else {}
    )

    with mlflow.start_run(run_name=run_name) as parent_run:
        mlflow.set_tags(
            {
                "stage": "baseline_benchmark",
                **tracking_tags,
            }
        )
        mlflow.log_params(
            {
                "cv_folds": cv_folds,
                "decision_threshold": threshold,
                "business_fn_cost": fn_cost,
                "business_fp_cost": fp_cost,
                "train_rows": X_train.shape[0],
                "holdout_rows": X_holdout.shape[0],
                "n_features": X_train.shape[1],
                "benchmark_model_count": len(models),
                "class_rate_train": float(y_train.mean()),
                "class_rate_holdout": float(y_holdout.mean()),
                "train_metric_sample_size": train_metric_sample_size,
            }
        )

        for artifact_path in extra_artifacts:
            artifact_file = Path(artifact_path)
            if artifact_file.exists():
                mlflow.log_artifact(str(artifact_file), artifact_path="tables")

        for model_name, model in models.items():
            metadata = catalog_lookup.get(model_name, {})
            family = metadata.get("family", "unknown")
            print(f"\n===== Baseline {model_name} =====")

            with mlflow.start_run(run_name=f"baseline_{model_name}", nested=True):
                mlflow.set_tags(
                    {
                        "model_name": model_name,
                        "model_family": family,
                        "benchmark_scope": metadata.get(
                            "benchmark_scope",
                            "full_dataset",
                        ),
                        "why_included": metadata.get("why_included", ""),
                        "library": (
                            "lightgbm"
                            if model_name == "lightgbm_bonus"
                            else "scikit-learn"
                        ),
                    }
                )
                mlflow.log_params(sanitize_model_params(model))

                folds_df, summary_df = evaluate_classifier_cv(
                    model=model,
                    X=X_train,
                    y=y_train,
                    cv=cv_folds,
                    threshold=threshold,
                    fn_cost=fn_cost,
                    fp_cost=fp_cost,
                    random_state=random_state,
                    train_metric_sample_size=train_metric_sample_size,
                )
                fitted_model, holdout_scores, holdout_metrics = evaluate_holdout(
                    model=model,
                    X_train=X_train,
                    y_train=y_train,
                    X_holdout=X_holdout,
                    y_holdout=y_holdout,
                    threshold=threshold,
                    fn_cost=fn_cost,
                    fp_cost=fp_cost,
                )

                train_scores = score_to_probability(fitted_model, X_train)
                train_metrics = classification_metrics_at_threshold(
                    y_true=y_train,
                    y_scores=train_scores,
                    threshold=threshold,
                    fn_cost=fn_cost,
                    fp_cost=fp_cost,
                )

                summary_flat = flatten_cv_summary(summary_df)

                # Alignement strict avec les clés de métriques du notebook 03.
                log_train_test_metrics(train_metrics, holdout_metrics)
                mlflow.log_metrics(
                    {
                        "cv_valid_business_cost_per_obs": float(
                            summary_flat.get("valid_business_cost_per_obs_mean", np.nan)
                        ),
                        "cv_valid_fit_time_seconds": float(
                            summary_flat.get("valid_fit_time_seconds_mean", np.nan)
                        ),
                    }
                )

                log_dataframe_artifact(
                    folds_df,
                    filename=f"{model_name}_cv_folds.csv",
                )
                log_dataframe_artifact(
                    summary_df,
                    filename=f"{model_name}_cv_summary.csv",
                )

                fitted_models[model_name] = fitted_model
                leaderboard_rows.append(
                    build_train_test_metric_row(
                        model_name=model_name,
                        family=family,
                        metadata=metadata,
                        threshold=threshold,
                        train_metrics=train_metrics,
                        test_metrics=holdout_metrics,
                        summary_flat=summary_flat,
                    )
                )

        baseline_leaderboard = pd.DataFrame(leaderboard_rows).sort_values(
            [
                "valid_business_cost_per_obs_mean",
                "test_business_fbeta",
                "valid_fit_time_seconds_mean",
                "test_pr_auc",
            ],
            ascending=[True, False, True, False],
        ).reset_index(drop=True)

        baseline_leaderboard["rank_valid_cost"] = baseline_leaderboard.index + 1
        baseline_leaderboard["rank_test_business_fbeta"] = (
            baseline_leaderboard["test_business_fbeta"]
            .rank(ascending=False, method="first")
            .astype(int)
        )

        baseline_leaderboard_path = processed_dir / "baseline_leaderboard.csv"
        baseline_leaderboard.to_csv(baseline_leaderboard_path, index=False)
        mlflow.log_artifact(str(baseline_leaderboard_path), artifact_path="tables")

        best_model_valid = baseline_leaderboard.iloc[0]["model_name"]
        best_model_test = (
            baseline_leaderboard.sort_values(
                "test_business_fbeta",
                ascending=False,
            )
            .iloc[0]["model_name"]
        )

        model_recommendation = {
            "best_model_on_valid_cost": best_model_valid,
            "best_model_on_test_business_fbeta": best_model_test,
            "model_retained_for_optimization": best_model_valid,
            "top_models_valid_cost": baseline_leaderboard["model_name"].tolist(),
            "top_models_test_business_fbeta": (
                baseline_leaderboard.sort_values(
                    "test_business_fbeta",
                    ascending=False,
                )["model_name"].tolist()
            ),
        }

        recommendation_path = processed_dir / "model_recommendation.json"
        recommendation_path.write_text(
            json.dumps(model_recommendation, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        mlflow.log_artifact(str(recommendation_path), artifact_path="tables")

        mlflow.log_params(
            {
                "best_model_on_valid_cost": best_model_valid,
                "best_model_on_test_business_fbeta": best_model_test,
                "model_retained_for_optimization": model_recommendation[
                    "model_retained_for_optimization"
                ],
            }
        )

        if (
            log_candidate_model_name is not None
            and log_candidate_model_name in fitted_models
        ):
            mlflow.sklearn.log_model(
                fitted_models[log_candidate_model_name],
                name="baseline_candidate_model",
            )

    print("Run parent MLflow :", parent_run.info.run_id)
    return baseline_leaderboard, model_recommendation, fitted_models

tracking_context = configure_mlflow_tracking(project_root)
tracking_uri = tracking_context.tracking_uri
experiment_name = tracking_context.experiment_name

print("Python executable :", sys.executable)
print("Tracking URI      :", tracking_uri)
print("Experiment        :", experiment_name)
print("UI command        : uv run mlflow ui --backend-store-uri", tracking_uri)
print("Train metric sample size :", train_metric_sample_size)


Python executable : c:\Users\kevin\Documents\P6\P6_MLOps_1-2\.venv\Scripts\python.exe
Tracking URI      : sqlite:///C:/Users/kevin/Documents/P6/P6_MLOps_1-2/mlflow.db
Experiment        : P6_HOME_CREDIT_DEFAULT_RISK
UI command        : uv run mlflow ui --backend-store-uri sqlite:///C:/Users/kevin/Documents/P6/P6_MLOps_1-2/mlflow.db
Train metric sample size : 20000


## Charger les données

On repart directement des fichiers `parquet` produits par le notebook 01.


In [15]:
train_full = pd.read_parquet(processed_dir / "application_train_full.parquet")
test_full = pd.read_parquet(processed_dir / "application_test_full.parquet")

X, y, X_test, quality_report = prepare_modeling_tables(
    train_df=train_full,
    test_df=test_full,
    id_col="SK_ID_CURR",
    target_col="TARGET",
    missing_threshold=0.95,
)

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=random_state,
    stratify=y,
)

print("X_train shape   :", X_train.shape)
print("X_holdout shape :", X_holdout.shape)
print("X_test shape    :", X_test.shape)
print("Taux de défaut train   :", round(float(y_train.mean()), 6))
print("Taux de défaut holdout :", round(float(y_holdout.mean()), 6))

quality_report_path = report_dir / "modeling_quality_report.csv"
quality_report.to_csv(quality_report_path, index=False)
display(quality_report.head(15))


X_train shape   : (246005, 733)
X_holdout shape : (61502, 733)
X_test shape    : (48744, 733)
Taux de défaut train   : 0.08073
Taux de défaut holdout : 0.080729


,feature,missing_ratio_train,dtype,nunique_train
533,refused_amt_down_payment_min,0.853116,float64,5273
534,refused_amt_down_payment_max,0.853116,float64,6494
535,refused_amt_down_payment_mean,0.853116,float64,9014
542,refused_rate_down_payment_min,0.853116,float64,11695
543,refused_rate_down_payment_max,0.853116,float64,15641
544,refused_rate_down_payment_mean,0.853116,float64,17145
532,refused_app_credit_perc_var,0.840140,float64,31695
638,cc_amt_payment_current_var,0.802870,float64,59041
673,cc_cnt_drawings_other_current_var,0.802629,float64,1499
623,cc_amt_drawings_other_current_var,0.802629,float64,4861


## Définir les modèles testés

On garde volontairement peu de modèles :

- `DummyClassifier` pour avoir une baseline naïve ;
- `SGDClassifier` pour une baseline linéaire plus adaptée au grand volume ;
- `HistGradientBoosting` pour un booster scikit-learn ;
- `LightGBM` comme baseline bonus et candidat naturel pour la suite.


In [16]:
lightgbm_device_type = resolve_lightgbm_device_type(random_state=random_state)
baseline_catalog = build_simple_baseline_catalog()
baseline_models = build_simple_baseline_models(
    random_state=random_state,
    lightgbm_device_type=lightgbm_device_type,
)

print("LightGBM device_type :", lightgbm_device_type)
display(baseline_catalog)
print("Modèles testés :", list(baseline_models.keys()))


LightGBM device_type : gpu


,model_name,family,benchmark_scope,why_included
0,dummy_classifier,Baseline naïve,full_dataset,référence minimale pour montrer qu'une forte a...
1,sgd_log_loss,Baseline linéaire,full_dataset,"baseline linéaire scalable, mieux adaptée que ..."
2,hist_gradient_boosting,Booster scikit-learn,full_dataset,"baseline tabulaire robuste, simple à comparer ..."
3,lightgbm_bonus,Booster bonus,full_dataset,benchmark bonus plus puissant et bon candidat ...


Modèles testés : ['dummy_classifier', 'sgd_log_loss', 'hist_gradient_boosting', 'lightgbm_bonus']


## Lancer le benchmark avec MLflow

Chaque modèle est évalué avec :

- une validation croisée stratifiée ;
- une métrique métier asymétrique avec `FN > FP` ;
- une évaluation holdout ;
- un logging clair dans MLflow.

Les métriques de référence affichées et logguées portent exactement
les mêmes noms que dans le notebook 03 :

- `train_accuracy` / `test_accuracy`
- `train_f1` / `test_f1`
- `train_business_fbeta` / `test_business_fbeta`
- `train_roc_auc` / `test_roc_auc`
- `train_pr_auc` / `test_pr_auc`


In [17]:
baseline_leaderboard, model_recommendation, fitted_models = run_simple_baseline_benchmark_with_mlflow(
    models=baseline_models,
    X_train=X_train,
    y_train=y_train,
    X_holdout=X_holdout,
    y_holdout=y_holdout,
    processed_dir=processed_dir,
    run_name="02_baselines_benchmark",
    cv_folds=cv_folds,
    threshold=0.5,
    fn_cost=business_fn_cost,
    fp_cost=business_fp_cost,
    random_state=random_state,
    train_metric_sample_size=train_metric_sample_size,
    model_catalog=baseline_catalog,
    tracking_tags={
        "notebook": "02_MODELISATION_BASELINES_MLFLOW",
        "lightgbm_device_type": lightgbm_device_type,
    },
    extra_artifacts=[quality_report_path],
    log_candidate_model_name="lightgbm_bonus",
)

leaderboard_display_columns = [
    "model_name",
    "family",
    "train_accuracy",
    "test_accuracy",
    "train_f1",
    "test_f1",
    "train_business_fbeta",
    "test_business_fbeta",
    "train_roc_auc",
    "test_roc_auc",
    "train_pr_auc",
    "test_pr_auc",
]
available_display_columns = [
    column for column in leaderboard_display_columns
    if column in baseline_leaderboard.columns
]

display(baseline_leaderboard[available_display_columns])
display(pd.DataFrame([model_recommendation]))



===== Baseline dummy_classifier =====

===== Baseline sgd_log_loss =====

===== Baseline hist_gradient_boosting =====

===== Baseline lightgbm_bonus =====


2026/04/17 12:21:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/17 12:21:42 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Run parent MLflow : 90abcfaab94d4d13b75aa4afd2bb56f0


,model_name,family,train_accuracy,test_accuracy,train_f1,test_f1,train_business_fbeta,test_business_fbeta,train_roc_auc,test_roc_auc,train_pr_auc,test_pr_auc
0,lightgbm_bonus,Booster bonus,0.758314,0.743927,0.344982,0.301689,0.639038,0.556560,0.853633,0.787233,0.365799,0.286739
1,sgd_log_loss,Baseline linéaire,0.676226,0.676466,0.256011,0.252461,0.527449,0.518351,0.741344,0.736234,0.202241,0.194601
2,hist_gradient_boosting,Booster scikit-learn,0.922957,0.920620,0.120143,0.085425,0.071070,0.050137,0.842664,0.785932,0.396401,0.284827
3,dummy_classifier,Baseline naïve,0.919270,0.919271,0.000000,0.000000,0.000000,0.000000,0.500000,0.500000,0.080730,0.080729


,best_model_on_valid_cost,best_model_on_test_business_fbeta,model_retained_for_optimization,top_models_valid_cost,top_models_test_business_fbeta
0,lightgbm_bonus,lightgbm_bonus,lightgbm_bonus,"[lightgbm_bonus, sgd_log_loss, hist_gradient_b...","[lightgbm_bonus, sgd_log_loss, hist_gradient_b..."


In [18]:
from IPython.display import Markdown, display

best_row = baseline_leaderboard.iloc[0]
second_row = baseline_leaderboard.iloc[1] if len(baseline_leaderboard) > 1 else None
linear_row = baseline_leaderboard.loc[
    baseline_leaderboard["model_name"] == "sgd_log_loss"
].iloc[0]
dummy_row = baseline_leaderboard.loc[
    baseline_leaderboard["model_name"] == "dummy_classifier"
].iloc[0]

conclusion_lines = [
    "## Conclusion",
    "",
    f"Le meilleur candidat sur ce benchmark est **{best_row['model_name']}**, "
    f"qui obtient un **F1 métier test de {best_row['test_business_fbeta']:.4f}**, "
    f"un **ROC AUC test de {best_row['test_roc_auc']:.4f}** "
    f"et une **PR AUC test de {best_row['test_pr_auc']:.4f}**.",
]

if second_row is not None:
    conclusion_lines.append(
        f"Le deuxième modèle, **{second_row['model_name']}**, reste proche avec "
        f"un **F1 métier test de {second_row['test_business_fbeta']:.4f}**, "
        f"ce qui en fait une alternative crédible mais légèrement moins performante."
    )

conclusion_lines.extend(
    [
        "",
        f"La baseline linéaire **sgd_log_loss** est plus rapide et plus simple, "
        f"mais elle reste nettement derrière "
        f"(F1 métier test = {linear_row['test_business_fbeta']:.4f}, "
        f"PR AUC test = {linear_row['test_pr_auc']:.4f}).",
        "",
        f"Le **dummy_classifier** rappelle pourquoi l'accuracy seule est trompeuse "
        f"dans un problème déséquilibré : malgré une accuracy test de "
        f"{dummy_row['test_accuracy']:.4f}, il obtient un **F1 test nul** "
        f"et un **F1 métier nul**, donc il n'est pas exploitable métier.",
        "",
        f"Pour la suite du projet, le modèle retenu pour l'optimisation est "
        f"**{model_recommendation['model_retained_for_optimization']}**.",
        "",
        "Le benchmark exporte à la fois un leaderboard simple au format "
        "`train_* / test_*` pour la comparaison directe avec le notebook 03, "
        "et les colonnes de pilotage CV nécessaires à la sélection du candidat."
    ]
)

display(Markdown("\n".join(conclusion_lines)))


## Conclusion

Le meilleur candidat sur ce benchmark est **lightgbm_bonus**, qui obtient un **F1 métier test de 0.5566**, un **ROC AUC test de 0.7872** et une **PR AUC test de 0.2867**.
Le deuxième modèle, **sgd_log_loss**, reste proche avec un **F1 métier test de 0.5184**, ce qui en fait une alternative crédible mais légèrement moins performante.

La baseline linéaire **sgd_log_loss** est plus rapide et plus simple, mais elle reste nettement derrière (F1 métier test = 0.5184, PR AUC test = 0.1946).

Le **dummy_classifier** rappelle pourquoi l'accuracy seule est trompeuse dans un problème déséquilibré : malgré une accuracy test de 0.9193, il obtient un **F1 test nul** et un **F1 métier nul**, donc il n'est pas exploitable métier.

Pour la suite du projet, le modèle retenu pour l'optimisation est **lightgbm_bonus**.

Le benchmark exporte à la fois un leaderboard simple au format `train_* / test_*` pour la comparaison directe avec le notebook 03, et les colonnes de pilotage CV nécessaires à la sélection du candidat.